# Sketch2Image — Stable Diffusion 1.5 + ControlNet

Run the cells from top to bottom.
- **Colab:** *Runtime → Change runtime type → T4 GPU*.
- **Kaggle:** *Settings → Accelerator → GPU T4*, and turn **Internet on**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Set your GitHub repository URL.

In [ ]:
REPO = "https://github.com/MarufDzine/sketch2image"
!git clone -q $REPO
%cd sketch2image

In [ ]:
!pip install -q -r requirements.txt

## 2. Sketches
Your sketches should be in `sketches/` (commit them to the repo), named as in `config.yaml` (`cat.jpeg`, `duck.jpeg`, `fish.jpeg`, `helicopter.jpeg`, `house.jpeg`).
**Colab only:** you can also upload them here instead, with the same file names.

In [ ]:
UPLOAD = False   # set True to upload sketches from your computer (Colab only)
if UPLOAD:
    from google.colab import files
    import shutil
    for name in files.upload():
        shutil.move(name, f"sketches/{name}")
!ls sketches

## 3. Check the preprocessing
This shows what the model actually sees for each sketch and each control type.

In [ ]:
import yaml, matplotlib.pyplot as plt
from sketch2img import prepare
cfg = yaml.safe_load(open("config.yaml"))
controls = ["scribble", "lineart", "canny"]
fig, axes = plt.subplots(len(cfg["sketches"]), 4, figsize=(12, 3.2 * len(cfg["sketches"])), squeeze=False)
for i, sk in enumerate(cfg["sketches"]):
    for j, c in enumerate(controls):
        img, lines, ctrl = prepare(sk["file"], c)
        if j == 0:
            axes[i, 0].imshow(img); axes[i, 0].set_title(sk["file"])
        axes[i, j + 1].imshow(ctrl); axes[i, j + 1].set_title(c)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Quick test: one image (~1–2 min the first time, because the models download)

In [ ]:
!python sketch2img.py --sketch sketches/cat.jpeg --prompt "a cute cat face, photorealistic, highly detailed" --out test.png
from IPython.display import Image, display
display(Image("test.png", width=400))

## 5. Run all experiments (~10–15 min on a T4)
If the session disconnects, re-run this cell. Images that already exist are skipped.

In [ ]:
!python run_experiments.py

In [ ]:
!python make_figures.py

## 6. Results

In [ ]:
import glob
from IPython.display import Image, display, Markdown
for f in sorted(glob.glob("outputs/figures/*.png")):
    display(Markdown(f"**{f}**")); display(Image(f, width=1000))

In [ ]:
import pandas as pd
pd.read_csv("outputs/tables/summary.csv")

## 7. Download everything
Push `outputs/` to your GitHub repo, and copy `outputs/tables/*.tex` into Overleaf.

In [ ]:
!zip -qr outputs.zip outputs
try:
    from google.colab import files; files.download("outputs.zip")
except ImportError:
    print("Kaggle: download outputs.zip from the Output panel on the right")

## 8. Web UI (bonus)
Click the `*.gradio.live` link that appears. Stop the cell to shut the UI down.

In [ ]:
!python app.py --share